## Tutorial: Extending 3' Exons in Arabidopsis Annotation

This notebook demonstrates how to load an Arabidopsis genome annotation from a GFF3 file, extend the 3' exons of transcripts by a specified length, and export the modified annotation.

This tutorial is particularly useful for analyzing 3' end sequencing data (3'-Seq), where the majority of reads map to the 3' end of transcripts. Standard genome annotation files (GFFs) often lack accurate 3' and 5' UTR annotations, as these regions are difficult to predict computationally without experimental evidence. By extending the 3' exons, we can better capture and analyze the full extent of 3' UTR regions that may not be properly annotated in the original GFF file.

### Step 1: Import Required Modules
We start by importing the necessary libraries: `aegis` for handling genomic annotations and `urlretrieve` from `urllib.request` to download the GFF3 file.

In [12]:
from aegis import *
from urllib.request import urlretrieve

### Step 2: Download and Load the Annotation
Download the Arabidopsis Araport11 GFF3 file from a remote URL and create an `Annotation` object named `a` to represent the genome annotation.

In [13]:
extra_3_exon = 500
gff_filename, _ = urlretrieve("https://raw.githubusercontent.com/Tomsbiolab/aegis/refs/heads/main/tests/test_data/input/annotation/arabidopsis_araport11.gff3")
a = Annotation(gff_filename, "Arabidopsis_annotation")


Processing Arabidopsis_annotation annotation object

gff3 file format detected for file='/tmp/tmpqp6_5bf2'

Creating Arabidopsis_annotation annotation object took 0.0 minutes

Correcting feature coordinates for Arabidopsis_annotation
Corrected feature coordinates for Arabidopsis_annotation

Whole update process for Arabidopsis_annotation annotation object took 0.0 minutes



### Step 3: Extend 3' Exons
Iterate through each chromosome, gene, and transcript in the annotation. For each transcript:
- If the strand is positive (`+`), add a new exon starting right after the last exon and extending by `extra_3_exon` (500 bp).
- If the strand is negative (`-`), add a new exon ending just before the first exon and extending backward by `extra_3_exon` (500 bp).
- Assign attributes to the new exon, linking it to the parent transcript.

In [14]:
for chrom, genes in a.chrs.items():
    for gid, g in genes.items():
        for tid, t in g.transcripts.items():
            if t.strand == "+":
                start = t.exons[-1].end + 1
                end = t.exons[-1].end + extra_3_exon
            elif t.strand == "-":
                start = t.exons[0].start - extra_3_exon
                end = t.exons[0].start - 1

            t.exons.append(Exon(feature_id="temp_id", ch=chrom, source=t.source, feature="exon", strand=t.strand, start=start, end=end, score=t.score, parents=[t.id]))

### Step 4: Update and Export the Annotation
Update the annotation to reflect changes, rename IDs for exons and UTRs, and export the modified annotation to a new GFF3 file, including UTRs.

In [ ]:
a.update()
a.rename_ids(features=["exon", "UTR"])
a.export.gff(tag="arabidopsis_extended_3.gff3", UTRs=True)

Correcting feature coordinates for Arabidopsis_annotation
Corrected feature coordinates for Arabidopsis_annotation

Whole update process for Arabidopsis_annotation annotation object took 0.0 minutes


Renaming Arabidopsis_annotation ids with prefix='', changing={'exon'} features took 0.0 minutes
Exporting Arabidopsis_annotation gff to out_gffs/arabidopsis_extended_3.gff3.
